# Stratifying a FlowModel in place

Build flows once, then `copy()` / `stratify()` / `adjust_flow()` instead of
re-declaring every transition on a new map. This notebook covers age and
strain stratification, adjustment precedence, Erlang latent stages, and
`Dest(...)` masks on dest-only properties.

The first curves are the unstratified epidemic and the same epidemic
after an even age split: total `I` should match. The rate bars are
adjustment precedence on one edge. The latent-period figures compare a
single `E` with an Erlang-3 chain. The last bars are a destination
multiplier: severe inflows should be double mild ones.


In [ ]:
import pandas as pd
import plotly.io as pio

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

from typing import NamedTuple

import numpy as np
import pytest

from summer4 import (
    Compartments,
    Dest,
    FlowModel,
    Multiply,
    Overwrite,
    Param,
    Property,
    PropertyMap,
    SavePlan,
    SaveRequest,
    Source,
    TraitChain,
    TransitionFlow,
)
from summer4.epi import ForceOfInfection, MixingMatrix


class P(NamedTuple):
    beta: float
    sigma: float
    gamma: float


state = Property("state", ("S", "E", "I", "R"))
pop = Property("pop", ("all",))
pmap = PropertyMap.from_property(state).stratify(pop)

mixing = MixingMatrix(pop, np.array([[1.0]]), check_reciprocal=False)
core = FlowModel(pmap)
core.add_flow(
    TransitionFlow(
        "infection",
        state["S"],
        state["E"],
        ForceOfInfection(
            "infection",
            infectious=state["I"],
            group_by=mixing.prop,
            mixing=mixing,
            kind="frequency",
            contact_rate=Param("beta"),
        ),
    )
)
core.add_flow(TransitionFlow("progression", state["E"], state["I"], Param("sigma")))
core.add_flow(TransitionFlow("recovery", state["I"], state["R"], Param("gamma")))
core.set_initial_population(
    {state["S"]: 990.0, state["E"]: 0.0, state["I"]: 10.0, state["R"]: 0.0}
)

params = P(beta=0.4, sigma=0.5, gamma=0.2)
plan = SavePlan(requests={"comp": SaveRequest(Compartments())}, ts=np.linspace(0.0, 40.0, 81))
core_cm = core.compile()
core_res = core_cm.run(params, t0=0.0, t1=40.0, dt=0.5, save=plan, solver="euler")
core_I = np.asarray(core_res["comp"].select(state["I"]).values.data).sum(axis=-1)
assert core_I.max() > 10.0

pd.DataFrame({"I": core_I}, index=np.linspace(0.0, 40.0, 81)).plot(
    title="Unstratified SEIR: infectious people",
    labels={"index": "time", "value": "people"},
)


## Homogeneous age stratification preserves aggregate I

`copy()` then `stratify(age)` with equal initial splits and unchanged mixing
(FOI still groups by `pop`) keeps the total infectious trajectory.
The two lines should lie on top of each other.


In [ ]:
age = Property("age", ("young", "old"))
aged = core.copy()
aged.stratify(age)
aged_cm = aged.compile()
aged_res = aged_cm.run(params, t0=0.0, t1=40.0, dt=0.5, save=plan, solver="euler")
aged_I = np.asarray(aged_res["comp"].select(state["I"]).values.data).sum(axis=-1)
np.testing.assert_allclose(aged_I, core_I, rtol=1e-5, atol=1e-5)
assert aged.pmap.size == 2 * core.pmap.size

pd.DataFrame(
    {"unstratified": core_I, "age-stratified total": aged_I},
    index=np.linspace(0.0, 40.0, 81),
).plot(
    title="Even age stratification does not change total I",
    labels={"index": "time", "value": "infectious people"},
)


## Adjustment precedence after a second stratification

An age `Multiply` declared before `stratify(strain)`, then an `Overwrite` on
strain b, yields $\beta_b \cdot a$ where both apply (overwrite is level 0).
Pass `precedence=2` on the overwrite to get summer2's "later wins" $\beta_b$.
The bars are the inflow on the old, strain-b edge: level-0 overwrite
still multiplies by the age factor; precedence 2 replaces the rate.


In [ ]:
strain = Property("strain", ("a", "b"))
beta_b = 0.25
age_mult = 2.0

strained = aged.copy()
strained.adjust_flow("infection", Multiply(age_mult, where=age["old"]))
strained.stratify(strain)
strained.adjust_flow("infection", Overwrite(beta_b, where=strain["b"]))

# Plain rate (not FOI) so per-edge rates are exactly the adjustment chain.
plain = FlowModel(strained.pmap)
plain.add_flow(
    TransitionFlow(
        "infection",
        state["S"],
        state["E"],
        1.0,
        adjust=list(strained.flows[0].adjust),
    )
)
pcm = plain.compile()
pflow = pcm.flows["infection"]
y = np.ones(pcm.pmap.size)
dy = np.asarray(pcm.vector_field(0.0, y, {}))
both = pflow.edge_map.mask(Source(age["old"])) & pflow.edge_map.mask(Source(strain["b"]))
assert both.any()
for i in np.flatnonzero(both):
    assert dy[int(pflow.dest_idx[i])] == pytest.approx(beta_b * age_mult)

# precedence=2 on the overwrite → summer2 "later wins" (β_b, not β_b·a).
summer2_style = FlowModel(strained.pmap)
summer2_style.add_flow(
    TransitionFlow(
        "infection",
        state["S"],
        state["E"],
        1.0,
        adjust=[
            Multiply(age_mult, where=age["old"]),
            Overwrite(beta_b, where=strain["b"], precedence=2),
        ],
    )
)
cm2 = summer2_style.compile()
pflow2 = cm2.flows["infection"]
dy2 = np.asarray(cm2.vector_field(0.0, y, {}))
both2 = pflow2.edge_map.mask(Source(age["old"])) & pflow2.edge_map.mask(Source(strain["b"]))
for i in np.flatnonzero(both2):
    assert dy2[int(pflow2.dest_idx[i])] == pytest.approx(beta_b)

i_both = int(np.flatnonzero(both)[0])
i_both2 = int(np.flatnonzero(both2)[0])
pd.Series(
    {
        "level-0 overwrite": float(dy[int(pflow.dest_idx[i_both])]),
        "precedence 2": float(dy2[int(pflow2.dest_idx[i_both2])]),
    }
).to_frame("inflow").plot.bar(
    title="Later overwrite wins only when its precedence is higher",
    labels={"index": "adjustment", "value": "people / time into E"},
)


## Same-level overwrite overlap raises

Two overwrites of the young band at the same precedence are ambiguous,
so `compile` raises. There is no trajectory to plot — the claim is the
exception, including the hint to set `precedence=`.


In [ ]:
overlap = FlowModel(aged.pmap)
overlap.add_flow(
    TransitionFlow(
        "infection",
        state["S"],
        state["E"],
        1.0,
        adjust=[
            Overwrite(0.0, where=age["young"]),
            Overwrite(0.5, where=age["young"]),
        ],
    )
)
try:
    overlap.compile()
    raised = False
except ValueError as exc:
    raised = True
    assert "Overlapping Overwrite" in str(exc)
    assert "precedence=" in str(exc)
assert raised

## Erlang latent stages via stratify + update_flow

Three E stages with rate $3\sigma$ keep the mean latent duration $1/\sigma$
while narrowing the time-to-I distribution. The epidemic peaks should
stay close. In the cohort — everyone starts in `E`, or in `e1` — the
Erlang chain should put fewer people in `I` at t = 1, because the
early exponential trickle is gone.


In [ ]:
stage = Property("stage", ("e1", "e2", "e3"))
erlang = core.copy()
erlang.stratify(stage, where=state["E"])
erlang.update_flow("infection", dest=state["E"] & stage["e1"])
erlang.update_flow("progression", source=state["E"] & stage["e3"])
erlang.adjust_flow("progression", 3.0)
erlang.add_flow(
    TransitionFlow(
        "latent_stages",
        state["E"],
        state["E"],
        Param("sigma"),
        pairing=TraitChain(stage, (("e1", "e2"), ("e2", "e3"))),
        adjust=[3.0],
    )
)

erlang_cm = erlang.compile()
erlang_res = erlang_cm.run(params, t0=0.0, t1=40.0, dt=0.5, save=plan, solver="euler")
erlang_I = np.asarray(erlang_res["comp"].select(state["I"]).values.data).sum(axis=-1)
# Same mean latent duration 1/σ → similar epidemic peak height.
np.testing.assert_allclose(float(erlang_I.max()), float(core_I.max()), rtol=0.05)

# Cohort: seed only E (e1 for Erlang) and measure early trickle vs mid-rise.
cohort_plan = SavePlan(
    requests={"comp": SaveRequest(Compartments())}, ts=np.linspace(0.0, 20.0, 201)
)
y_core = np.asarray(core_cm.initial_state(params).data).copy()
y_core[:] = 0.0
y_core[core_cm.pmap.select(state["E"])] = 1000.0
core_cohort = core_cm.run(
    params, y_core, t0=0.0, t1=20.0, dt=0.1, save=cohort_plan, solver="euler"
)
y_erl = np.asarray(erlang_cm.initial_state(params).data).copy()
y_erl[:] = 0.0
y_erl[erlang_cm.pmap.select(state["E"] & stage["e1"])] = 1000.0
erl_cohort = erlang_cm.run(
    params, y_erl, t0=0.0, t1=20.0, dt=0.1, save=cohort_plan, solver="euler"
)
core_I_c = np.asarray(core_cohort["comp"].select(state["I"]).values.data).sum(axis=-1)
erl_I_c = np.asarray(erl_cohort["comp"].select(state["I"]).values.data).sum(axis=-1)
# Erlang is less variable: less I by t=1 (early trickle).
assert float(erl_I_c[10]) < float(core_I_c[10])
assert float(erl_I_c[80]) > 100.0  # still progresses on the 1/σ timescale

pd.DataFrame(
    {"one latent compartment": core_I, "Erlang-3": erlang_I},
    index=np.linspace(0.0, 40.0, 81),
).plot(
    title="Same mean latent period: the epidemic peaks stay close",
    labels={"index": "time", "value": "infectious people"},
)
pd.DataFrame(
    {"one latent compartment": core_I_c, "Erlang-3": erl_I_c},
    index=np.linspace(0.0, 20.0, 201),
).plot(
    title="A cohort in E: Erlang delays the early rise in I",
    labels={"index": "time", "value": "infectious people"},
)


## Dest(...) on a severity-only destination property

Severity exists only on `I`. A multiplier on `Dest(severity['severe'])`
doubles the inflow to severe and leaves mild alone. The bars are those
two destination derivatives.


In [ ]:
severity = Property("severity", ("mild", "severe"))
sev = FlowModel(PropertyMap.from_property(state).stratify(pop))
sev.add_flow(TransitionFlow("infection", state["S"], state["I"], 1.0))
sev.stratify(severity, where=state["I"])
sev.adjust_flow("infection", Multiply(2.0, where=Dest(severity["severe"])))
scm = sev.compile()
sflow = scm.flows["infection"]
dy = np.asarray(scm.vector_field(0.0, np.ones(scm.pmap.size), {}))
severe = set(scm.pmap.select(state["I"] & severity["severe"]).tolist())
mild = set(scm.pmap.select(state["I"] & severity["mild"]).tolist())
for dest_i, weight in zip(sflow.dest_idx, sflow.weight, strict=True):
    d = int(dest_i)
    if d in severe:
        assert dy[d] == pytest.approx(2.0 * float(weight))
    elif d in mild:
        assert dy[d] == pytest.approx(1.0 * float(weight))

pd.Series(
    {
        "mild dest": float(dy[next(iter(mild))]),
        "severe dest": float(dy[next(iter(severe))]),
    }
).to_frame("inflow").plot.bar(
    title="Dest(severity) doubles only the severe inflow",
    labels={"index": "destination", "value": "people / time"},
)
